<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/04_validation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Chargement**

In [1]:
import os, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

NPY_DIR = "/content/drive/MyDrive/OTTO_Recommender_System/data/processed"
N, N_SESSIONS = 216_716_096, 12_899_779

aid_arr   = np.load(f"{NPY_DIR}/aid.npy")
ts_arr    = np.load(f"{NPY_DIR}/ts.npy")
type_arr  = np.load(f"{NPY_DIR}/type.npy")
sizes_arr = np.load(f"{NPY_DIR}/sizes.npy")
assert len(aid_arr) == N and len(sizes_arr) == N_SESSIONS and sizes_arr.sum() == N
print("Chargé")

Mounted at /content/drive
Chargé


**seuil T et sessions de validation**

In [2]:
cum      = np.concatenate([[0], np.cumsum(sizes_arr, dtype=np.int64)])
first_ts = ts_arr[cum[:-1]]
last_ts  = ts_arr[cum[1:] - 1]

ts_end = (int(ts_arr.max()) // 3600 + 1) * 3600
T = ts_end - 7 * 86400
print("Fin du train :", pd.to_datetime(ts_end, unit="s"))
print("Seuil T      :", pd.to_datetime(T, unit="s"))

val_ids  = np.where((first_ts >= T) & (sizes_arr >= 2))[0]
straddle = ((first_ts < T) & (last_ts >= T)).sum()
print(f"Sessions de validation        : {len(val_ids):,}  (test : 1 671 803)")
print(f"Sessions à cheval sur T       : {straddle:,}")
print(f"Événements avant T / après T  : {(ts_arr < T).sum():,} / {(ts_arr >= T).sum():,}")

Fin du train : 2022-08-28 22:00:00
Seuil T      : 2022-08-21 22:00:00
Sessions de validation        : 1,801,251  (test : 1 671 803)
Sessions à cheval sur T       : 3,521,833
Événements avant T / après T  : 163,955,181 / 52,760,915


**labels (avec test unitaire)**

In [3]:
def split_labels(suf_aid, suf_type):
    """Cibles à partir des événements APRÈS la coupure."""
    clicks = suf_aid[suf_type == 0]
    return {"clicks": {int(clicks[0])} if len(clicks) else set(),   # 1 seul : le prochain click
            "carts":  set(map(int, suf_aid[suf_type == 1])),
            "orders": set(map(int, suf_aid[suf_type == 2]))}

# test unitaire : à comprendre ligne par ligne
toy_aid  = np.array([7, 8, 8, 9, 7])
toy_type = np.array([0, 0, 1, 1, 2])
assert split_labels(toy_aid, toy_type) == {"clicks": {7}, "carts": {8, 9}, "orders": {7}}
print("Test labels OK")

def build_dev(val_ids, n_dev, seed=42):
    rng = np.random.default_rng(seed)
    dev = rng.choice(val_ids, size=n_dev, replace=False)
    GT, prefix_len = {}, {}
    for sid in dev:
        a, z = cum[sid], cum[sid + 1]
        k = int(rng.integers(1, z - a))              # 1 <= k <= n-1
        GT[int(sid)] = split_labels(aid_arr[a + k:z], type_arr[a + k:z])
        prefix_len[int(sid)] = k
    return GT, prefix_len

t0 = time.time()
GT, prefix_len = build_dev(val_ids, n_dev=300_000)
print(f"Jeu de dev construit en {time.time()-t0:.0f}s")

Test labels OK
Jeu de dev construit en 7s


**métrique et tests de sanité**

In [4]:
WEIGHTS = {"clicks": 0.10, "carts": 0.30, "orders": 0.60}

def weighted_recall(preds, GT, k=20):
    """preds[type][session] = liste d'aids (max 20). Micro-average, comme la compétition."""
    out = {}
    for t in WEIGHTS:
        hits = denom = 0
        for sid, g in GT.items():
            gt = g[t]
            if not gt:                       # pas de label pour ce type : ignorée
                continue
            hits  += len(set(preds[t].get(sid, [])[:k]) & gt)
            denom += min(k, len(gt))
        out[t] = hits / denom if denom else 0.0
    out["score"] = sum(WEIGHTS[t] * out[t] for t in WEIGHTS)
    return out

# Test 1 : l'oracle doit donner 1.0
oracle = {t: {s: list(g[t])[:20] for s, g in GT.items()} for t in WEIGHTS}
print("Oracle :", weighted_recall(oracle, GT))

# Test 2 : prédictions vides -> 0.0
print("Vide   :", weighted_recall({t: {} for t in WEIGHTS}, GT))

# Test 3 : calcul à la main (1 session, 1 click, 3 carts, pas d'orders)
toy_gt = {1: {"clicks": {5}, "carts": {1, 2, 3}, "orders": set()}}
r = weighted_recall({"clicks": {1: [5, 6]}, "carts": {1: [1, 2]}, "orders": {1: [9]}}, toy_gt)
assert abs(r["score"] - (0.10 * 1.0 + 0.30 * 2/3)) < 1e-9
print("Test à la main OK :", r)

Oracle : {'clicks': 1.0, 'carts': 1.0, 'orders': 1.0, 'score': 1.0}
Vide   : {'clicks': 0.0, 'carts': 0.0, 'orders': 0.0, 'score': 0.0}
Test à la main OK : {'clicks': 1.0, 'carts': 0.6666666666666666, 'orders': 0.0, 'score': 0.3}


**calibration face au test**

In [5]:
t_size = np.load(f"{NPY_DIR}/test_sizes.npy")
pl = np.array(list(prefix_len.values()))
cmp = pd.DataFrame({"dev (préfixe)": pd.Series(pl).describe(percentiles=[.5, .75, .95]),
                    "test (visible)": pd.Series(t_size).describe(percentiles=[.5, .75, .95])})
display(cmp.round(2))

for t in WEIGHTS:
    n_lab = sum(1 for g in GT.values() if g[t])
    print(f"{t:<7}: {n_lab:,} sessions avec au moins un label ({n_lab/len(GT)*100:.1f} %)")

,dev (préfixe),test (visible)
count,300000.00,1671803.00
mean,4.29,4.14
std,8.52,8.22
min,1.00,1.00
50%,2.00,2.00
75%,4.00,4.00
95%,16.00,15.00
max,422.00,458.00


clicks : 292,456 sessions avec au moins un label (97.5 %)
carts  : 51,038 sessions avec au moins un label (17.0 %)
orders : 25,073 sessions avec au moins un label (8.4 %)
